**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# State-Space Models: Kalman → S4 → Mamba

The course only a signal processing society can teach properly. Modern sequence architectures (S4, Mamba) are *literally* this curriculum: state-space models ([Kalman](./Intro_AdFilt_KF.ipynb)), convolution kernels ([Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb)), and discretization ([Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)) — rebranded for deep learning. Four sessions from the linear SSM you already know to a trained sequence model, with every identity verified numerically.

## 1. Pre-requisites

- [Kalman](./Intro_AdFilt_KF.ipynb) & [RNN](./Intro_RNN.ipynb) workshops.
- [Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) for the architecture being challenged.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Linear SSMs Are Convolutions* (~35 min)
**Goal:** prove (numerically) that an LTI state space = one long FIR filter; why that unlocks parallel training.
**Builds on:** [Kalman](./Intro_AdFilt_KF.ipynb). &nbsp; **Feeds into:** Session 2 (HiPPO & discretization).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Linear SSMs Are Convolutions</b></summary>

**Timing (~35 min).** 5 min framing the whole workshop · 12 min the unrolling derivation · 8 min the oracle check · 8 min why the identity is worth so much · 2 min buffer.

**Open with the thesis of the workshop.** S4 and Mamba are usually taught as new deep learning architectures. They are not new to this room: they are state-space models, convolution kernels, and discretization — every one of which this curriculum has already covered under its DSP name. Say plainly that the goal is to make the papers readable as signal processing literature. That framing is why this workshop exists, and stating it up front changes how students hear the next two hours.

**Derive the unrolling on the board; it is four lines.** Start from $h_t = \bar A h_{t-1} + \bar B x_t$ with $h_{-1} = 0$. Substitute once, then again, and let the room see the pattern: $h_t = \sum_k \bar A^k \bar B x_{t-k}$. Apply $C$ and the kernel $K_k = C\bar A^k \bar B$ falls out. Nobody should be told this identity — everyone should watch it emerge, because the entire workshop is downstream of it.

**Name what the identity buys, in both directions.** Train as a convolution: fully parallel across time, FFT-accelerable, and no backpropagation through time, so no vanishing-gradient path. Infer as a recurrence: $O(1)$ state per step, no growing KV cache. The room has felt both pains — the [RNN workshop](./Intro_RNN.ipynb) for the first, [transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) for the second. This one identity dodges both, and that is the whole architectural bet.

**Misconception.** "So an SSM is just an RNN written differently." Only for *linear* dynamics. The equivalence requires no nonlinearity between steps — that is precisely what lets the recurrence be unrolled into a fixed kernel. An LSTM's per-step nonlinearity destroys it. Students who miss this cannot understand why Session 4's selectivity is a genuine sacrifice rather than a free upgrade, so plant it now and call back to it later.

**Ask the room.** "The kernel has length $T$. Isn't a convolution with a length-$T$ kernel expensive?" $O(T^2)$ directly — but $O(T\log T)$ by FFT, which is the [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) fast-convolution machinery doing load-bearing work in a 2023 architecture. Compare against attention's $O(T^2)$, which has no such escape.

**Point at the plot.** The kernel is a sum of geometric decays — the poles of the system are its modes, exactly as in [Complex Analysis](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb). It dies within a few dozen taps here, and that observation is the entire setup for Session 2. Do not fix it yet; let them see the problem first.

**If the demo misbehaves.** The `assert` at 1e-9 is generous; agreement should be at machine precision. Failure means `A` was made non-diagonal or unstable — eigenvalues outside the unit circle make $\bar A^k$ blow up and the kernel diverge.
</details>

## 2. The Identity Everything Rests On

💡 **Intuition.** A linear time-invariant SSM $h_{t} = \bar{A} h_{t-1} + \bar{B} x_t, \; y_t = C h_t$ can be *unrolled*: $y_t = \sum_{k\ge0} C\bar{A}^{k}\bar{B} \, x_{t-k}$ — a *convolution* with kernel $K_k = C \bar A^k \bar B$. That one identity is the whole trick: **train as a convolution** (parallel, FFT-fast, no backprop-through-time vanishing) and **infer as a recurrence** (constant memory per step, unlike attention's growing KV cache). RNN pain and transformer pain, both dodged — for *linear* state dynamics.

In [2]:
# ORACLE CHECK: recurrence output == convolution output, elementwise
d_state, T = 8, 200
A = np.diag(rng.uniform(0.7, 0.98, d_state))          # stable diagonal SSM
Bm = rng.standard_normal((d_state, 1))
Cm = rng.standard_normal((1, d_state))
x = rng.standard_normal(T)

# path 1: run the recurrence
h = np.zeros(d_state); y_rec = np.zeros(T)
for t in range(T):
    h = A @ h * 1.0 + (Bm[:, 0] * x[t])
    y_rec[t] = Cm[0] @ h

# path 2: materialize the kernel, convolve
K = np.array([ (Cm @ np.linalg.matrix_power(A, k) @ Bm)[0, 0] for k in range(T) ])
y_conv = np.convolve(x, K)[:T]

print("max |recurrence − convolution| =", np.abs(y_rec - y_conv).max())
assert np.abs(y_rec - y_conv).max() < 1e-9
plt.figure(figsize=(7.5, 2.2)); plt.plot(K[:60], ".-")
plt.title("the SSM's implicit FIR kernel  $K_k = C\\bar{A}^k\\bar{B}$")
plt.tight_layout(); plt.show()

max |recurrence − convolution| = 1.7763568394002505e-15


/tmp/ipykernel_2953716/2656292720.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two completely different computations — a sequential loop over 200 timesteps carrying a hidden state, and a single convolution against a precomputed 200-tap kernel — produced the same output to **1.8e-15**. That is machine precision, so the identity $y_t = \sum_k C\bar A^k\bar B\,x_{t-k}$ is exact rather than approximate, and the `assert` guarantees it stays that way.

**Why this is the most important cell in the workshop.** The two paths have opposite computational profiles, and the identity lets you pick whichever you need:

- **Training** wants the convolution. Every output is computed in parallel, there is no sequential dependency across time, and the whole thing runs by FFT in $O(T\log T)$. Critically, there is no backpropagation *through time* — the gradient does not thread through 200 sequential multiplications, so the vanishing-gradient pathology that limits the [RNN workshop](./Intro_RNN.ipynb) simply has no path to occur.
- **Inference** wants the recurrence. Generating token by token, you carry a fixed-size state $h$ and pay $O(1)$ per step — against a transformer's KV cache, which grows linearly in context and dominates memory at long sequence lengths.

Getting both from one model is the architectural bet of the whole S4 family, and it rests entirely on this equality.

**Read the kernel plot, because it sets up Session 2.** $K_k = C\bar A^k \bar B$ with diagonal $\bar A$ is a weighted sum of geometric decays $\lambda_i^k$ — the eigenvalues are the system's poles, and each contributes a mode dying at its own rate. Here the $\lambda_i$ were drawn uniformly from $[0.7, 0.98]$, and the kernel is visibly dead within a few dozen taps. An SSM whose kernel has collapsed by $k = 50$ cannot possibly use information from 400 steps ago, whatever its parameter count. That is the vanishing-memory problem in linear, fully visible form, and Session 2 is about fixing it.

**The load-bearing assumption.** This identity holds only because the dynamics are *linear and time-invariant*: the same $\bar A$, $\bar B$, $C$ at every step. Insert any per-step nonlinearity, as an LSTM does, and the unrolling is impossible. Remember this when Session 4 makes the matrices input-dependent — the convolution path is exactly what Mamba gives up, and it is worth knowing now what is being spent.

---
### 🕐 Session 2 of 4 — *HiPPO & Discretization: Why S4's A Matrix Is Special* (~35 min)
**Goal:** see why random A forgets; meet the memory-optimal initialization and the continuous-time view.
**Builds on:** Session 1; [Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb). &nbsp; **Feeds into:** Session 3 (training an SSM).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: HiPPO & Discretization</b></summary>

**Timing (~35 min).** 10 min why random $A$ forgets · 10 min the timescale-spread fix · 8 min the demo · 7 min discretization and learnable $\Delta$.

**Board first — reduce it to one mode.** Take a single pole: the kernel contribution is $\lambda^k$, so the memory horizon is roughly $1/(1-\lambda)$ steps. Have the room compute it for $\lambda = 0.9$ (about 10 steps) and $\lambda = 0.999$ (about 1000). Now ask what happens when you draw 32 poles uniformly from $[0.7, 0.95]$: every one of them has a horizon of a few dozen steps, so the *sum* also dies in a few dozen steps. Averaging many short memories does not produce a long one. That single observation is the session.

**The reframe that makes HiPPO make sense.** Long memory is not a capacity problem or a training problem — it is an **initialization** problem. The architecture was always capable of remembering 400 steps; the poles simply were not placed where that is possible. Students expect "add more parameters" or "train longer" to be the answer, and neither is. Say explicitly that this is a case where *where you start* determines what is reachable.

**What HiPPO actually is, stated honestly.** The real construction chooses $A$ so the state holds coefficients of an orthogonal-polynomial expansion of the input history — provably the optimal compression of the past under a chosen measure. We are not implementing that. Our `np.exp(-np.logspace(...))` is a *caricature* that keeps the property that matters here: timescales spread across decades, so some modes are fast and some are extremely slow. Be clear about the substitution, or students will think HiPPO is just log-spacing.

**Ask the room.** "Why spread timescales rather than just set every pole close to 1?" Because a model with only slow modes cannot represent anything fast — it low-passes the input and loses local structure entirely. You need both, and the spread is what gives the network a *choice* of horizon per channel. This also previews Session 4: fixed spread means fixed choices, and selectivity is about making the choice per-token.

**Discretization deserves its five minutes.** $\bar A = e^{A\Delta}$ is [Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)'s continuous-to-discrete map, and it is the same mathematics as choosing a sampling rate. The striking part is that $\Delta$ is *learnable*, per channel: the network learns its own sampling rate, so some channels look at fine detail while others see a heavily decimated view. Framed that way — a learned multi-rate filter bank — it is a DSP idea the room already owns.

**Point at the number, not just the plots.** The `memory_span` helper is the honest measurement: last tap still above 1% of peak. Random $A$ gives 47 steps; decade-spread gives 399+, where the `+` means it never fell below the threshold within the window computed. Do not claim the second is "eight times better" — it is *not measured*, which the debrief says plainly.
</details>

## 3. Long Memory Is an Initialization Problem

💡 **Intuition.** Session 1's kernel is a sum of geometric decays $\lambda_i^k$ ([Complex Analysis](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb): the poles are the modes!). Random stable $A$ ⇒ all modes die at similar rates ⇒ effective memory of a few dozen steps, the RNN disease in linear form. **HiPPO**'s insight: choose $A$ so the state stores *orthogonal-polynomial coefficients of the input's history* — a principled spread of timescales, kernels with long structured tails. S4 = HiPPO-initialized continuous SSM, discretized ([FoSP2 S2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)'s $\bar{A} = e^{A\Delta}$, in practice bilinear/ZOH) with a *learnable* step size $\Delta$ — the network literally learns its own sampling rate per channel.

In [3]:
# Kernel shapes: random-diagonal vs HiPPO-style log-spaced timescales
def kernel_of(eigs, T=400):
    C1 = np.ones(len(eigs)) / len(eigs)                # equal weights: shape comes from the poles
    return np.array([ (C1 * eigs**k).sum() for k in range(T) ])

eig_rand = rng.uniform(0.7, 0.95, 32)
eig_hippo_ish = np.exp(-np.logspace(-3.5, 0, 32))     # timescales spread over decades

fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.6))
axes[0].plot(kernel_of(eig_rand)); axes[0].set_title("random A: kernel dead by k≈50")
axes[1].plot(kernel_of(eig_hippo_ish)); axes[1].set_title("decade-spread timescales: structure for hundreds of steps")
plt.tight_layout(); plt.show()
def memory_span(K):                                    # last step where the kernel is still >1% of its peak
    return int(np.max(np.nonzero(np.abs(K) > 0.01 * np.abs(K).max())))
print(f"kernel span (last tap above 1% of peak): random A → {memory_span(kernel_of(eig_rand))} steps,"
      f"  decade-spread → {memory_span(kernel_of(eig_hippo_ish))}+ steps")

kernel span (last tap above 1% of peak): random A → 47 steps,  decade-spread → 399+ steps


/tmp/ipykernel_2953716/1097700265.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Same architecture, same number of modes, same equal weighting — only the *placement of the poles* differs. Random poles drawn from $[0.7, 0.95]$ give a kernel whose last significant tap is at step **47**. Poles spread over decades give one still above 1% of its peak at step **399**, the end of the window.

Read that `399+` carefully: the `+` means the kernel never dropped below the threshold inside the range we computed, so 399 is a *lower bound* on the span, not a measurement of it. The honest statement is "at least eight times longer, and we did not find the end." That distinction matters more than it looks, because the whole claim of this session is about reaching horizons we have not bounded.

**Why the random version fails is arithmetic, not mystery.** A single pole $\lambda$ contributes $\lambda^k$, with a memory horizon of roughly $1/(1-\lambda)$ steps. Drawing 32 poles uniformly from $[0.7, 0.95]$ gives horizons between about 3 and 20 steps — every mode is short, so their sum is short too. You cannot build a long memory by averaging many short ones. The decade-spread version deliberately includes poles like $e^{-10^{-3.5}} \approx 0.99968$, whose horizon is thousands of steps, alongside fast ones that preserve local detail.

**The reframe worth carrying away.** Long memory here is not a matter of capacity or training effort — both models have identical capacity, and neither was trained at all. It is an **initialization** problem: the architecture could always represent a 400-step dependency, but random initialization placed every pole where that is unreachable, and gradient descent starting from a dead kernel has no signal telling it to move poles toward 1. This is why HiPPO is a contribution at all.

**Be clear on what we did and did not implement.** Real HiPPO derives $A$ from an orthogonal-polynomial expansion of the input's history, provably optimal for compressing the past under a chosen measure. Our `np.exp(-np.logspace(-3.5, 0, 32))` is a caricature that keeps only the property doing the work in this demo — timescales spread over decades. It reproduces the *shape* of the benefit, not the theory that motivates the specific matrix.

**And what discretization adds.** In the continuous view the poles come from $\bar A = e^{A\Delta}$, so the step size $\Delta$ sets where they land. Making $\Delta$ learnable per channel means the network chooses its own sampling rate — a learned multi-rate filter bank, in [Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)'s language. Session 3's layer parameterises exactly this, via `log_dt` initialised on `torch.linspace(-4, 0, ...)`: the decade spread, made trainable.

---
### 🕐 Session 3 of 4 — *Train a Diagonal SSM* (~40 min)
**Goal:** build an S4-style layer (diagonal, conv-trained) and beat the LSTM on a long-memory task.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (selectivity & Mamba).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Train a Diagonal SSM</b></summary>

**Timing (~40 min).** 10 min walking the layer · 8 min the FFT-vs-recurrence oracle · 15 min the bake-off (training takes a couple of minutes — start it early and talk over it) · 7 min reading the result honestly.

**Walk the layer in four moves, not line by line.** (1) `log_dt` parameterises the poles as $\lambda = e^{-e^{\theta}}$, which keeps them in $(0,1)$ for *any* real $\theta$ — so gradient descent can never accidentally produce an unstable system. That double exponential is a real design decision, not obfuscation, and it is worth a minute. (2) `C` is the learnable output mixing. (3) `kernel()` materialises $K$ from the poles. (4) `forward()` convolves by FFT with `n=2*T` zero-padding, which makes the circular FFT convolution compute a *linear* one — the overlap-save concern from [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb), and a good place to ask why the padding is there.

**The oracle cell is the one to dwell on.** It re-verifies Session 1's identity, but now on the *actual trainable layer* rather than a toy — FFT-convolution path against the naive recurrence, agreeing to 2.4e-6. Say why the tolerance dropped from 1e-15 to 1e-4: this is float32, and the FFT accumulates round-off across the transform. That is not a defect, it is the precision you are working in, and knowing which tolerance is appropriate for which dtype is a genuinely useful engineering habit.

**Set up the task before running it.** A ±1 cue at $t=0$, then 400 steps of pure noise, then classify. There is no partial credit and no shortcut: any model that has forgotten position 0 is at chance. Ask the room to predict what an LSTM will do before you run it — most will expect it to struggle, and being right builds confidence for the harder claims in Session 4.

**Frame the bake-off honestly — this is the important instruction.** The SSM reaches 93.4% and the LSTM sits at 54.7%, barely above chance. Do *not* present this as "SSMs beat LSTMs." It is one synthetic task, 300 training steps, one seed, and no hyperparameter search for either model. What it demonstrates is an *optimisation* difference on a long-range dependency: the SSM's gradient reaches $t=0$ through a convolution in one hop, while the LSTM's must traverse 400 sequential steps. A fairer characterisation is "the SSM learns this in 300 steps; the LSTM does not," which is a real and relevant claim without overreaching. The debrief states this explicitly — do not undercut it in the room.

**Ask the room.** "Why is the LSTM near chance rather than merely worse?" Because this task has no gradient signal until the cue is actually used — there is no partial solution to hill-climb toward. Either the dependency is learned or the model outputs the class prior. Long-range tasks often have this all-or-nothing structure, which is exactly why they are the benchmark for this architecture family.

**If the demo misbehaves.** Both models are small and the run is short; the SSM occasionally lands in the 80s and the LSTM occasionally drifts to 60% on a lucky seed. The qualitative gap is robust, the exact numbers are not. If the SSM fails outright, check that `log_dt` still initialises on `linspace(-4, 0)` — collapse that spread and Session 2's lesson reasserts itself immediately, which is a fine impromptu demonstration.
</details>

## 4. The Layer, Assembled

Our layer (an honest simplification of S4D): per channel, learnable log-timescales $\lambda = e^{-e^{\theta}}$, learnable $C$; compute the kernel, convolve via FFT, add a skip and a nonlinearity. Trained **as a convolution**, verified equal to its recurrence.

In [4]:
class DiagSSMLayer(nn.Module):
    def __init__(self, d_model=32, d_state=32, T_max=1024):
        super().__init__()
        self.log_dt = nn.Parameter(torch.linspace(-4, 0, d_state).repeat(d_model, 1))  # decade spread
        self.C = nn.Parameter(torch.randn(d_model, d_state) / d_state**0.5)
        self.D = nn.Parameter(torch.ones(d_model))                                     # skip
        self.T_max = T_max
    def kernel(self, T):
        lam = torch.exp(-torch.exp(self.log_dt))              # (d_model, d_state) in (0,1)
        k = torch.arange(T)
        K = torch.einsum("ds,dsk->dk", self.C, lam[:, :, None] ** k[None, None, :])
        return K                                               # (d_model, T)
    def forward(self, x):                                      # x: (B, T, d_model)
        B, T, D = x.shape
        K = self.kernel(T)
        Xf = torch.fft.rfft(x.transpose(1, 2), n=2*T)          # convolve per channel via FFT
        Kf = torch.fft.rfft(K, n=2*T)
        y = torch.fft.irfft(Xf * Kf[None], n=2*T)[..., :T].transpose(1, 2)
        return torch.nn.functional.gelu(y + x * self.D)

# ORACLE: layer's FFT-conv path == naive recurrence, for one channel
layer = DiagSSMLayer(d_model=4, d_state=8)
x_test = torch.randn(1, 64, 4)
with torch.no_grad():
    y_fft = layer(x_test)
    lam = torch.exp(-torch.exp(layer.log_dt))
    y_rec = torch.zeros_like(x_test)
    for d in range(4):
        h = torch.zeros(8)
        for t in range(64):
            h = lam[d] * h + x_test[0, t, d]
            y_rec[0, t, d] = (layer.C[d] * h).sum()
    y_rec = torch.nn.functional.gelu(y_rec + x_test * layer.D)
print("max |FFT-conv − recurrence| =", (y_fft - y_rec).abs().max().item())
assert (y_fft - y_rec).abs().max() < 1e-4

max |FFT-conv − recurrence| = 2.384185791015625e-06


**What just happened.** Session 1's identity, re-checked on the real trainable layer rather than a toy: the FFT-convolution forward pass and a hand-written recurrence over the same parameters agree to **2.4e-6**. The layer we are about to train really does have both computational forms.

**The tolerance moved, and the reason is worth naming.** Session 1 agreed to 1.8e-15; here the `assert` allows 1e-4 and we land at 2.4e-6. Nothing degraded — that is the difference between float64 and float32, compounded by an FFT that accumulates round-off across a length-128 transform. Knowing which tolerance is appropriate for which dtype is a practical skill: asserting 1e-15 on float32 would fail on correct code, and asserting 1e-4 on float64 would pass on broken code.

**What the parameterisation is doing.** The poles are $\lambda = e^{-e^{\theta}}$, a double exponential that maps any real $\theta$ into $(0,1)$. That means gradient descent *cannot* produce an unstable system no matter how large a step it takes — stability is enforced by the parameterisation rather than by clipping or a penalty. Compare with the alternative of learning $\lambda$ directly and projecting it back into range: this is cleaner, has no discontinuity in the gradient, and is a nice example of buying a hard guarantee through a change of variables.

**One detail that is easy to miss.** The FFT uses `n=2*T`, zero-padding to double length. Without it the FFT would compute a *circular* convolution, wrapping the end of the sequence around to contaminate the beginning — the model would appear to see the future. The padding is what makes it a linear convolution, and it is the same overlap-save concern from [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) appearing inside a modern architecture.

With the equivalence confirmed, the next cell trains through the convolution path — parallel across all 400 timesteps, with gradients that never traverse a 400-step sequential chain — while the same weights could be run as an $O(1)$-per-step recurrence at inference time. That is the payoff the workshop has been building toward.

In [5]:
# The long-memory gauntlet: recall the FIRST token's class after T=400 noise steps
def task_batch(B=64, T=400):
    x = torch.randn(B, T, 1) * 0.3
    labels = torch.randint(0, 2, (B,))
    x[:, 0, 0] = labels.float() * 2 - 1                       # ±1 cue at t=0, then noise
    return x, labels

class SSMNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.inp = nn.Linear(1, 32)
        self.s1, self.s2 = DiagSSMLayer(32), DiagSSMLayer(32)
        self.head = nn.Linear(32, 2)
    def forward(self, x):
        h = self.inp(x); h = self.s1(h); h = self.s2(h)
        return self.head(h[:, -1])

class LSTMNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 32, 2, batch_first=True)
        self.head = nn.Linear(32, 2)
    def forward(self, x):
        return self.head(self.lstm(x)[0][:, -1])

results = {}
for name, model in [("SSM", SSMNet()), ("LSTM", LSTMNet())]:
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    accs = []
    for step in range(300):
        x, yb = task_batch()
        loss = nn.functional.cross_entropy(model(x), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 25 == 0:
            with torch.no_grad():
                xv, yv = task_batch(256)
                accs.append((model(xv).argmax(1) == yv).float().mean().item())
    results[name] = accs
    print(f"{name}: final recall accuracy over 400 steps = {accs[-1]:.1%}")

plt.figure(figsize=(7.5, 2.6))
for name, accs in results.items(): plt.plot(np.arange(len(accs))*25, accs, "o-", label=name)
plt.axhline(0.5, color="k", linestyle=":", linewidth=0.8, label="chance")
plt.legend(); plt.xlabel("training step"); plt.ylabel("val accuracy")
plt.title("remember one token across 400 steps: decade-spread SSM vs LSTM")
plt.tight_layout(); plt.show()

SSM: final recall accuracy over 400 steps = 93.4%


LSTM: final recall accuracy over 400 steps = 54.7%


/tmp/ipykernel_2953716/2672149721.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Remember one token across 400 steps of noise: the SSM reaches **93.4%**, the LSTM **54.7%** against a chance floor of 50%. Both models are small, both trained for 300 steps with the same optimizer and learning rate, on the same batches.

**Why the task is all-or-nothing.** A ±1 cue at $t=0$, then 400 steps of pure noise carrying no information whatsoever, then a binary classification. There is no partial credit and nothing to hill-climb toward: a model that has lost position 0 can do no better than output the class prior. This is why the LSTM sits *at chance* rather than merely underperforming — it is not a slightly worse solution, it is the absence of one.

**The mechanism, which is the actual lesson.** The SSM's gradient reaches $t=0$ through a *convolution* — one hop, no sequential chain, and the kernel tap at lag 400 is a directly-parameterised quantity that Session 2 made sure is non-negligible. The LSTM's gradient must traverse 400 sequential steps, each multiplying by a Jacobian, and the product of 400 such factors either vanishes or explodes long before it reaches the cue. Sessions 1 and 2 are both load-bearing here: Session 1's identity gives the short gradient path, Session 2's timescale spread ensures there is a live kernel tap at lag 400 to carry it.

**Now the honest caveats, because the headline invites overreach.** This is *one synthetic task, one seed, 300 training steps, and no hyperparameter search for either model*. It does not establish that SSMs beat LSTMs in general, and it is not a benchmark result. A carefully tuned LSTM given far longer, or gradient clipping and a better initialisation, would do better than 54.7% — the literature on long-range LSTM training is substantial. What this cell legitimately shows is an **optimisation** difference on a long-range dependency: the SSM finds this solution easily and quickly, and the LSTM does not find it within a comparable budget. That is a real and practically relevant claim, and it is narrower than "SSMs are better."

It is also worth noting what the SSM does *not* do here. 93.4% is not 100%, and the residual errors come from the model having to distinguish the cue from noise of comparable amplitude at a single timestep. The architecture solved the memory problem; the remaining gap is a signal-detection problem, which is a different thing.

**And the limitation that motivates Session 4.** This layer is linear and time-invariant — the same kernel applies to every input. It cannot decide to remember one token and ignore another, because it has no mechanism that depends on content. That works perfectly here, where the important token is always at a fixed position. It fails as soon as *which* token matters depends on what the tokens are.

---
### 🕐 Session 4 of 4 — *Selectivity & Mamba (Frontier Sketch)* (~30 min)
**Goal:** understand what Mamba changes — input-dependent dynamics — and what that costs.
**Builds on:** Session 3.

---

<details>
<summary>🎓 <b>Teacher notes — Session 4: Selectivity & Mamba</b></summary>

**Timing (~30 min).** 8 min the limitation of LTI · 8 min what selectivity changes and what it costs · 8 min the toy demo · 6 min the summary table and close.

**Set expectations first — this is a frontier sketch.** The ℹ️ banner is not boilerplate. We explain the mechanism and its trade-off; we do not implement Mamba's hardware-aware selective scan, and the toy below uses an honest $O(T)$ Python loop. Say this at the start, so nobody leaves thinking they have seen Mamba implemented.

**Board first — expose the limitation as a question.** Sessions 1–3 built a system with one fixed kernel applied to every input. Ask: what task would that fail at? Steer toward anything where *which* token matters depends on *what* the tokens are — "return the word after the marker," where the marker's position varies. A fixed kernel weights by position only, so it cannot express "attend to whatever follows the marker." Attention can, via content-based routing, and that is the capability gap selectivity closes.

**State the trade-off in this course's own terms, because it is the payoff of the whole workshop.** Making $\bar B, \bar C, \Delta$ depend on the input makes the system **time-varying**. Session 1's unrolling required time-invariance — the same $\bar A$ at every step is what let $\bar A^k$ appear as a fixed kernel. So the convolution identity *is destroyed*, and with it the FFT training path. Ask the room to state this before you do; they have everything needed to derive it, and it is far more satisfying than being told.

**Then the resolution.** Mamba's contribution is not the idea of input-dependent gating — gated RNNs have done that since the 1990s. It is showing the resulting recurrence can still be computed fast on a GPU, via a parallel associative scan with kernel fusion to keep the state in SRAM. That is the [HW-Accelerated](../Intro_GPU/HW_Accelerated_Computing.ipynb) toolbox being the actual contribution of a famous ML paper — a point worth making to a room that may think architecture research is purely mathematical.

**Misconception.** "Selectivity is strictly better, so Mamba supersedes S4." It is a trade, not an upgrade. You give up the FFT convolution and take on a much harder implementation; in exchange you get content-dependent routing. For tasks where position determines relevance — many signal processing problems, notably — the LTI model is simpler, faster, and entirely sufficient. Reach for selectivity when content decides what matters.

**On the toy's numbers.** MSE 0.0078 against a predict-the-mean baseline of 0.0839, roughly 11× better. The baseline matters: without it, 0.0078 is an uninterpretable number. Make sure students see *why* the variance of the target is the right null model — it is what you achieve knowing nothing about the input.

**Close the workshop on the table.** Walk the four columns and have the room supply the DSP name for each: nonlinear IIR, data-adaptive kernel, FIR bank with learned poles, time-varying system. That is the moment the thesis lands — these architectures are signal processing, and the room can now read the papers as such.
</details>

## 5. What Mamba Adds — and What It Breaks

> ℹ️ **Frontier sketch.** This session explains the mechanism and its trade-off; a full efficient selective-scan implementation (Mamba's hardware-aware kernel) is beyond a 40-minute session and is *not* implemented here.

💡 **Intuition.** Everything above is **LTI**: the same kernel for every input — the system cannot *decide* to remember this token and forget that one. Mamba makes $\bar B, \bar C, \Delta$ **functions of the current input** ('selective'): a content-controlled gate on the state, per step. The price is exactly the trade this course's structure predicts: input-dependent dynamics are **time-varying**, so the convolution identity of Session 1 *no longer holds* — no FFT training path. Mamba's contribution is showing the recurrence can still be computed fast on GPUs (parallel associative scan + kernel fusion — the [HW-Accelerated](../Intro_GPU/HW_Accelerated_Computing.ipynb) toolbox earning its keep).

The one-line summary of the whole architecture family:

| | RNN/LSTM | Transformer | S4 (LTI SSM) | Mamba (selective) |
|---|---|---|---|---|
| Train | sequential | parallel | parallel (FFT conv) | parallel (assoc. scan) |
| Infer/step | $O(1)$ | $O(T)$ (KV cache) | $O(1)$ | $O(1)$ |
| Content-dependent routing | gates | **attention** | ✗ | **selection** |
| DSP name | nonlinear IIR | data-adaptive kernel | FIR bank w/ learned poles | time-varying system |

In [6]:
# The selectivity mechanism in miniature (correct but naive O(T) loop — the SKETCH):
# gate Δ_t = f(x_t) controls how much the state updates on each token
class SelectiveToy(nn.Module):
    def __init__(self, d=16):
        super().__init__()
        # the gate sees the current token AND the previous one (a 1-step conv, as in Mamba's
        # local conv before the SSM) — Δ depends on the INPUT: this is the selectivity
        self.gate = nn.Linear(2, d)
        self.C = nn.Parameter(torch.randn(d) / d**0.5)
    def forward(self, x):                  # x: (B, T, 1)
        B, T, _ = x.shape
        x_prev = torch.cat([torch.zeros(B, 1, 1), x[:, :-1]], 1)
        feats = torch.cat([x, x_prev], -1)
        dt = torch.sigmoid(self.gate(feats))   # (B, T, d) in (0,1): per-token write strength
        h = torch.zeros(B, dt.shape[-1])
        ys = []
        for t in range(T):                 # honest recurrence — no conv shortcut EXISTS here
            h = (1 - dt[:, t]) * h + dt[:, t] * x[:, t]
            ys.append(h @ self.C)
        return torch.stack(ys, 1)

# task LTI SSMs cannot do: output the last token that FOLLOWED a '2' marker
toy = SelectiveToy(); opt = torch.optim.Adam(toy.parameters(), lr=1e-2)
def sel_batch(B=128, T=40):
    x = torch.rand(B, T, 1)
    pos = torch.randint(5, T-1, (B,))
    x[torch.arange(B), pos, 0] = 2.0                       # marker
    target = x[torch.arange(B), pos+1, 0]                  # remember what came after it
    return x, target
for step in range(600):
    x, tgt = sel_batch()
    loss = ((toy(x)[:, -1] - tgt)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
x, tgt = sel_batch(512)
with torch.no_grad(): mse = ((toy(x)[:, -1] - tgt)**2).mean().item()
print(f"selective toy MSE on marker-recall: {mse:.4f}  (predicting the mean would give ≈{tgt.var().item():.4f})")
print("→ an input-dependent gate solves a task no fixed kernel can — that's Mamba's bet, at scale")

selective toy MSE on marker-recall: 0.0078  (predicting the mean would give ≈0.0839)
→ an input-dependent gate solves a task no fixed kernel can — that's Mamba's bet, at scale


**What just happened.** MSE **0.0078** on marker-recall, against **0.0839** for predicting the mean — about 11× better than knowing nothing. The baseline is what makes the number readable: 0.0839 is the target's variance, which is exactly the error you achieve by ignoring the input entirely, so the model has genuinely extracted the token following a marker whose position varies from sample to sample.

**No LTI system can do this, and it is worth being precise about why.** Sessions 1–3 produced a fixed kernel: the output is $\sum_k K_k\,x_{t-k}$, weighting purely by *lag*. But the marker lands anywhere in $[5, T{-}1]$, so the informative token is at a different offset every time. A fixed kernel would have to weight all those offsets, which averages the correct token together with dozens of irrelevant ones. The failure is representational — no amount of training fixes it, because the function being asked for is not in the class.

The gate is what changes that. `dt = torch.sigmoid(self.gate(feats))` makes the write strength a function of the *current and previous token*, so the model can learn "when the previous token was a 2, write hard; otherwise hold." The state update $h \leftarrow (1-\delta_t)h + \delta_t x_t$ then behaves as content-controlled memory rather than a fixed decay. That is selectivity, and it is the whole of Mamba's conceptual step.

**And here is the bill, which Session 1 lets us state exactly.** Because $\delta_t$ depends on the input, the dynamics are **time-varying** — a different $\bar A$ at every step. Session 1's unrolling required a *single* $\bar A$ so that $\bar A^k$ could be a fixed kernel. That assumption is now false, so the convolution identity does not hold, and there is no FFT training path. Look at the code: the loop over `t` is not laziness, it is the honest computation, and the comment saying no conv shortcut *exists* is literally true.

So the trade is exact: content-dependent routing bought at the cost of parallel convolutional training. Mamba's actual contribution is making that affordable anyway — a parallel associative scan with kernel fusion that keeps the state in GPU SRAM, so the sequential-looking recurrence still saturates the hardware. The famous architecture paper is, in large part, a [GPU systems](../Intro_GPU/HW_Accelerated_Computing.ipynb) paper.

**What this toy is not.** It is a 16-dimensional gate on 40-step sequences with an $O(T)$ Python loop, trained on a task designed to need selectivity. It demonstrates the mechanism and its cost; it is not Mamba, and it says nothing about how the idea scales. Treat it as the smallest complete example of the principle — which, for a 30-minute session, is the right thing to have.

## 6. Conclusion

Linear SSM = convolution (verified), long memory = timescale spread (HiPPO's gift), training = FFT, inference = recurrence — and Mamba trades the conv identity for content-selective dynamics computed by scan. You can now read the S4/Mamba papers as *signal processing literature*, because that's what they are.

---
## Where next

- [LLMs from the Ground Up](../Intro_Mach_Learn/LLMs_from_the_Ground_Up.ipynb) — the model family SSMs compete with.
- [Kalman](./Intro_AdFilt_KF.ipynb) / [FoSP2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb) — the two halves this course glued together.
- [Modern Architectures](../Intro_Mach_Learn/Modern_Architectures.ipynb) — where SSM blocks sit in today's model zoo.